[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnrozhkov/santa-scale-challenge/blob/serverless/notebooks/02_image_to_video.ipynb)

# Image → video (Wan)

Playground for the **video** role: Wan I2V via `adapter_for("video", settings)`.
Pass PNG bytes + a motion prompt; you get MP4 bytes (`generate`). The adapter also
exposes `submit` / `poll` for the async path.

This notebook does **not** need a live image endpoint: the still is a 1×1 PNG
(or swap in a card from notebook 01). Run the generate cell only when a video
endpoint or OpenAI fallback is configured.

**Fallback** (`SANTA_FALLBACK`, default `auto`): try Wan first; on missing URL/token
or `AdapterError`, switch to OpenAI `sora-2`. `off` / `only` as usual.

**Secrets.** Locally `.env`; on Colab userdata named like `.env.example`:
`VIDEO_ENDPOINT_URL`, `VIDEO_ENDPOINT_TOKEN`, optionally `OPENAI_API_KEY`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def apply_colab_secrets() -> None:
    """Copy Colab userdata into os.environ. Secret names match .env.example."""
    try:
        from google.colab import userdata
    except ImportError:
        return
    for key in (
        "IMAGE_ENDPOINT_URL",
        "IMAGE_ENDPOINT_TOKEN",
        "VIDEO_ENDPOINT_URL",
        "VIDEO_ENDPOINT_TOKEN",
        "AUDIO_ENDPOINT_URL",
        "AUDIO_ENDPOINT_TOKEN",
        "TOKEN_FACTORY_API_KEY",
        "OPENAI_API_KEY",
        "SANTA_FALLBACK",
    ):
        try:
            value = userdata.get(key)
        except Exception:
            continue
        if value:
            os.environ[key] = str(value)


try:
    import google.colab  # noqa: F401
except ImportError:
    pass
else:
    apply_colab_secrets()
    repo = Path("/content/santa-scale-challenge")
    if not (repo / "pyproject.toml").is_file():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                "serverless",
                "--depth",
                "1",
                "https://github.com/mnrozhkov/santa-scale-challenge.git",
                str(repo),
            ]
        )
    os.chdir(repo)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)])

from santa.config import Settings

settings = Settings.load()
print("SANTA_FALLBACK =", os.environ.get("SANTA_FALLBACK", "auto"))
print(
    "env present:",
    [
        k
        for k in (
            "IMAGE_ENDPOINT_URL",
            "IMAGE_ENDPOINT_TOKEN",
            "VIDEO_ENDPOINT_URL",
            "VIDEO_ENDPOINT_TOKEN",
            "AUDIO_ENDPOINT_URL",
            "AUDIO_ENDPOINT_TOKEN",
            "TOKEN_FACTORY_API_KEY",
            "OPENAI_API_KEY",
        )
        if os.environ.get(k)
    ],
)


## Parameters

- **`MOTION`** — camera/snow prompt. `None` loads `config/prompts.yaml` via
  `santa.animate.load_prompts` / `motion_prompt`.
- **`SEED`** — optional int on `generate(..., seed=)`.
- **size, num_frames, fps, flow_shift, …** — `roles.video.options` in
  `config/models.yaml` (on `Settings`). Tweak with `video.cfg.options["size"] = "832x480"`
  and friends before calling `generate`.


In [ ]:
import io

from IPython.display import Image, display
from PIL import Image as PILImage

# 1px stand-in so this cell runs offline. Replace with a real card PNG for a useful clip:
# png = Path("out/<kid-id>/card.png").read_bytes()
tiny = PILImage.new("RGB", (1, 1), (255, 200, 80))
buf = io.BytesIO()
tiny.save(buf, format="PNG")
png = buf.getvalue()
display(Image(data=png, format="png"))
print("png bytes:", len(png))


In [ ]:
from IPython.display import Video, display
from santa.animate import load_prompts, motion_prompt
from santa.models import adapter_for

MOTION = None  # or a string; None → prompts.yaml
SEED = None

prompts = load_prompts()
motion = motion_prompt(prompts, MOTION)
video = adapter_for("video", settings)
# video.cfg.options["size"] = "832x480"
# video.cfg.options["num_frames"] = 81
# video.cfg.options["fps"] = 16

mp4 = video.generate(motion, image=png, seed=SEED)
clip = Path("out/notebook_i2v.mp4")
clip.parent.mkdir(parents=True, exist_ok=True)
clip.write_bytes(mp4)

print(video.cfg.label, "fallback=", video.used_fallback, "bytes=", len(mp4))
display(Video(str(clip), embed=True, width=640))
